# Solutions · 07 · Packed towers and mass transfer

Worked solutions to the exercises of [notebook 07](../notebooks/07_packed_towers_and_mass_transfer.ipynb). Try each exercise
yourself before reading its solution - the learning happens in the attempt. Solutions are one way to
answer each question; other correct approaches exist.

In [1]:
try:                      # on Google Colab (or anywhere engsep is missing): install it from GitHub
    import engsep
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ktwyw/engsep"], check=True)
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from engsep import absorption as ab
from engsep import transport as tr
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 90, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

import inspect
from IPython.display import Code

def show_source(obj):
    """Display the source code of a library function or class."""
    return Code(inspect.getsource(obj), language="python")

def heading(text):
    print(f"\n{text}\n" + "=" * len(text))

## Exercise 1

A packed tower absorbs acetone (2 %) from air into water (m = 2.0) with 95 % recovery at A = 1.6; G_M = 0.04 kmol/(m2 s), K_y a = 0.06 kmol/(m3 s). Find the height. Then halve the gas flux (a wider tower): how does the height change, and why less than proportionally (assume K_y a ~ G^0.7)?

In [2]:
m, y_in, rec, A = 2.0, 0.02, 0.95, 1.6
y_out = y_in * (1 - rec)
N_OG = ab.ntu_colburn(y_in, y_out, m, 0.0, A)
for G_M, K_y_a in ((0.04, 0.06), (0.02, 0.06 * 0.5 ** 0.7)):
    H_OG = ab.htu_gas(G_M, K_y_a)
    print(f"G_M = {G_M} kmol/(m2 s): H_OG = {H_OG:.2f} m, N_OG = {N_OG:.2f}, height = {ab.packed_height(H_OG, N_OG):.2f} m")

G_M = 0.04 kmol/(m2 s): H_OG = 0.67 m, N_OG = 5.59, height = 3.72 m
G_M = 0.02 kmol/(m2 s): H_OG = 0.54 m, N_OG = 5.59, height = 3.03 m


Halving the gas flux halves the numerator of the HTU but also reduces $K_ya$ (which grows roughly as
$G^{0.7}$), so the height falls by only about 20 %, not 50 %. A wider tower helps less than one might hope -
and its diameter follows from flooding, not from this calculation.

## Exercise 2

For a gas with m = 5 and k_y = 0.1, k_x = 0.5, what fraction of the resistance is in the liquid? By how much must k_x be raised (more liquid turbulence) to halve the total resistance?

In [3]:
tf = ab.overall_from_film(k_y=0.1, k_x=0.5, m=5.0)
R_total = 1 / tf["K_y"]
print(f"resistances: gas 1/k_y = {1/0.1:.0f}, liquid m/k_x = {5.0/0.5:.0f}, total {R_total:.0f} -> fraction in the liquid {1 - tf['gas_resistance_fraction']:.0%}")
for factor in (2, 5, 10):
    R_new = 1 / 0.1 + 5.0 / (0.5 * factor)
    print(f"k_x x {factor}: total resistance {R_new:.0f} ({R_new/R_total:.0%} of the original)")
print("halving the total would need the liquid resistance removed entirely: unreachable by any increase of k_x")

resistances: gas 1/k_y = 10, liquid m/k_x = 10, total 20 -> fraction in the liquid 50%
k_x x 2: total resistance 15 (75% of the original)
k_x x 5: total resistance 12 (60% of the original)
k_x x 10: total resistance 11 (55% of the original)
halving the total would need the liquid resistance removed entirely: unreachable by any increase of k_x


With $m$ = 5 the liquid film holds half the resistance; halving the total would need the liquid resistance
removed entirely, which no realistic increase in turbulence achieves. Two-film arithmetic tells you the
ceiling of an improvement before the money is spent.

## Exercise 3

**Implement it yourself:** compute Fuller's diffusivity for ammonia in air from the diffusion volumes and molar masses in `transport.FULLER_V` and `transport.MOLAR_MASS`, and compare with the library and with the measured 0.28 cm2/s.

In [4]:
MA, MB = tr.MOLAR_MASS["NH3"], tr.MOLAR_MASS["air"]
VA, VB = tr.FULLER_V["NH3"], tr.FULLER_V["air"]
T, P = 298.15, 101325.0
D = 1.013e-2 * T ** 1.75 * np.sqrt(1 / MA + 1 / MB) / (P * (VA ** (1 / 3) + VB ** (1 / 3)) ** 2)
print(f"by hand: {D*1e4:.3f} cm2/s; library {tr.fuller('NH3', 'air', T)*1e4:.3f}; measured 0.28")

by hand: 0.245 cm2/s; library 0.245; measured 0.28


Fuller's correlation from its diffusion volumes gives ammonia in air to within about 10 % of the measured
value - the accuracy the method is known for.